# Inspect what the pipeline produced

Open this after running `panmvpa-run --stage maps`. Every cell prints or plots
something — nothing computes silently.

Set `DATA_DIR` before launching Jupyter if your data isn't in the default place:

```bash
export DATA_DIR=$HOME/data/ds006598
```

**A quarter of the data** is block size 4 of 16 chunks; **all the data** is block 16.

In [ ]:
import sys, pathlib

# Make the package importable whether or not it was pip-installed.
repo = pathlib.Path.cwd()
if not (repo / "panmvpa").exists():
    repo = repo.parent
sys.path.insert(0, str(repo))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from panmvpa import config, parcellation, identify, rest

SUBJECT = "PAN01"
QUARTER = (0, 4)                      # first quarter of the rest data
FULL = (0, config.N_CHUNKS)           # all of it
QUARTER_SPECS = [(s, 4) for s in range(0, config.N_CHUNKS, 4)]   # Q1..Q4

print("data dir :", config.DATA_ROOT)
print("maps dir :", config.MAPS_DIR)
print("levels   :", config.LEVELS)
print()

stored = sorted(config.MAPS_DIR.glob(f"sub-{SUBJECT}_*.npy")) if config.MAPS_DIR.exists() else []
print(f"{len(stored)} stored maps for {SUBJECT}")
for p in stored:
    print("   ", p.name)
if not stored:
    print("\n!! No maps found. Run:  panmvpa-run --stage maps --subjects", SUBJECT)

## 1. See the maps: a quarter of the data vs all of it

Same slice, same colours. Each colour is one of the 17 networks. Look for borders that
move and for speckle that cleans up as data increases.

In [ ]:
SLICE_Z = 45          # axial slice index; change to look elsewhere

have_both = parcellation.has_map(SUBJECT, QUARTER) and parcellation.has_map(SUBJECT, FULL)
if not have_both:
    print("Need both maps. Missing:",
          [config.map_key(s) for s in (QUARTER, FULL) if not parcellation.has_map(SUBJECT, s)])
else:
    vols = {}
    for name, spec in [("quarter (4/16)", QUARTER), ("all data (16/16)", FULL)]:
        labels = parcellation.load_map(SUBJECT, spec)
        vol = np.zeros(parcellation.group_networks().shape, dtype=float)
        idx = parcellation.analysis_domain()
        vol[idx[0], idx[1], idx[2]] = labels
        vols[name] = vol

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, (name, vol) in zip(axes, vols.items()):
        sl = np.ma.masked_where(vol[:, :, SLICE_Z] == 0, vol[:, :, SLICE_Z])
        ax.imshow(np.rot90(sl), cmap="tab20", vmin=1, vmax=17, interpolation="nearest")
        ax.set_title(name)
        ax.axis("off")

    a, b = (vols["quarter (4/16)"], vols["all data (16/16)"])
    diff = ((a != b) & (a > 0)).astype(float)
    sl = np.ma.masked_where(diff[:, :, SLICE_Z] == 0, diff[:, :, SLICE_Z])
    axes[2].imshow(np.rot90(sl), cmap="autumn", interpolation="nearest")
    axes[2].set_title("voxels that CHANGED")
    axes[2].axis("off")
    plt.tight_layout(); plt.show()

    dom = parcellation.analysis_domain().shape[1]
    changed = int((parcellation.load_map(SUBJECT, QUARTER)
                   != parcellation.load_map(SUBJECT, FULL)).sum())
    print(f"whole brain: {changed:,} of {dom:,} voxels changed network "
          f"({100*changed/dom:.1f}%)")

## 2. How big is each network, at a quarter vs at full?

If a network collapses or explodes as data grows, that shows up here.

In [ ]:
if have_both:
    names = parcellation.network_order()
    q = parcellation.load_map(SUBJECT, QUARTER)
    f = parcellation.load_map(SUBJECT, FULL)
    group = parcellation.group_network_sizes()

    sizes = pd.DataFrame({
        "network": names,
        "group atlas": [group[n] for n in names],
        "quarter": [int((q == k).sum()) for k in range(1, 18)],
        "full": [int((f == k).sum()) for k in range(1, 18)],
    })
    sizes["change"] = sizes["full"] - sizes["quarter"]
    display(sizes)

    ax = sizes.set_index("network")[["quarter", "full"]].plot(
        kind="bar", figsize=(12, 4), width=0.8)
    ax.set_ylabel("voxels"); ax.set_title(f"{SUBJECT}: network size, quarter vs full")
    plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()

## 3. Dice matrix between the four quarter maps (Q1..Q4)

Four maps, each from a different quarter of the rest data, same amount of data each.
Off-diagonal values are the stability numbers that feed the top panel of the figure.

In [ ]:
missing = [config.map_key(s) for s in QUARTER_SPECS if not parcellation.has_map(SUBJECT, s)]
if missing:
    print("missing quarter maps:", missing)
else:
    qmaps = [parcellation.load_map(SUBJECT, s) for s in QUARTER_SPECS]
    labels = [f"Q{i+1}" for i in range(4)]
    mat = pd.DataFrame(
        [[parcellation.map_dice(a, b) for b in qmaps] for a in qmaps],
        index=labels, columns=labels,
    )
    print(f"{SUBJECT}: Dice between quarter-sized maps\n")
    display(mat.round(3))

    off = mat.values[np.triu_indices(4, k=1)]
    print(f"\n6 off-diagonal pairs: {np.round(off, 3).tolist()}")
    print(f"mean off-diagonal Dice = {off.mean():.3f}   <- the 4/16 stability point")

## 4. Dice per network at each level

The mean can hide things — Dice runs lower for small networks, so a rising mean could in
principle be driven by the big ones. Every network should rise down the columns.

In [ ]:
names = parcellation.network_order()
group = parcellation.group_network_sizes()
table = {}

for block in config.STABILITY_BLOCKS:
    pairs = config.stability_pairs(block)
    usable = [(a, b) for a, b in pairs
              if parcellation.has_map(SUBJECT, a) and parcellation.has_map(SUBJECT, b)]
    if not usable:
        continue
    per = np.array([
        parcellation.dice_per_network(parcellation.load_map(SUBJECT, a),
                                      parcellation.load_map(SUBJECT, b))
        for a, b in usable
    ])
    table[config.level_name(block)] = np.nanmean(per, axis=0)

if not table:
    print("No level has both maps of a pair yet.")
else:
    df = pd.DataFrame(table, index=names)
    df.insert(0, "group voxels", [group[n] for n in names])
    df = df.sort_values("group voxels")
    print(f"{SUBJECT}: Dice per network (rows sorted small -> large network)\n")
    display(df.round(3))

    lv = [c for c in df.columns if c != "group voxels"]
    print("\nmean over the 17 networks:")
    for c in lv:
        print(f"  {c:>6}: {df[c].mean():.3f}")

    ax = df[lv].T.plot(figsize=(11, 5), marker="o", legend=False, alpha=0.75)
    df[lv].mean().plot(ax=ax, marker="s", lw=3, color="black", label="mean")
    ax.set_ylabel("Dice"); ax.set_xlabel("data level")
    ax.set_title("Each line is one network — do they all rise?")
    ax.legend(fontsize=7, ncol=2, loc="upper left"); plt.tight_layout(); plt.show()

## 5. One held-out scan against every candidate map

The actual numbers behind an identification. Needs task scans on disk and maps for at
least two subjects. `margin` = correct subject's score minus the best wrong one.

In [ ]:
scans = rest.task_scans(SUBJECT)
lineup = parcellation.cohort(FULL)
print(f"held-out task scans for {SUBJECT}: {len(scans)}")
print(f"candidate maps in the lineup at 16/16: {sorted(lineup)}")

if not scans:
    print("\nNo task scans on disk — fetch with:  fetch_hub.py --kind task")
elif len(lineup) < 2:
    print("\nNeed maps for >=2 subjects to run a lineup.")
else:
    path = scans[0]
    print(f"\nscoring: {path.name}")
    ts = rest.load_scan(path)
    res = identify.identify(ts, lineup, SUBJECT)

    scored = pd.Series(res["scores"]).sort_values(ascending=False).to_frame("homogeneity")
    scored["is truth"] = [i == SUBJECT for i in scored.index]
    display(scored.round(5))

    print(f"\npredicted : {res['predicted']}")
    print(f"truth     : {res['true']}")
    print(f"correct   : {res['correct']}")
    print(f"margin    : {res['margin']:+.5f}   (positive = correct)")
    rest.clear_cache()

In [ ]:
# Margin across every held-out scan, if you want the whole distribution.
if scans and len(lineup) >= 2:
    rows = []
    for p in scans:
        r = identify.identify(rest.load_scan(p), lineup, SUBJECT)
        rows.append({"scan": p.name.split("_space")[0], "task": rest.task_name(p),
                     "predicted": r["predicted"], "correct": r["correct"],
                     "margin": r["margin"]})
        rest.clear_cache()
    df = pd.DataFrame(rows).sort_values("margin")
    display(df.round(5))
    print(f"accuracy    : {df['correct'].mean():.3f}")
    print(f"mean margin : {df['margin'].mean():+.5f}")
    df.plot(x="scan", y="margin", kind="barh", figsize=(8, 0.3 * len(df) + 2),
            legend=False, title="margin per held-out scan (negative = misidentified)")
    plt.axvline(0, color="k", lw=1); plt.tight_layout(); plt.show()